## Imports and db setup

In [2]:
import sys
from pathlib import Path
import duckdb
import zipfile
import pandas as pd



data_dir = Path("../data").resolve()
assert data_dir.is_dir(), f"Data folder not found: {data_dir}"

con = duckdb.connect(str(data_dir / "project.duckdb"))

print("Database:", data_dir / "project.duckdb")
print(con.sql("SELECT version()").fetchall()[0][0])

Database: C:\Users\marks\.workspace\projects\data4600-project\code\data\project.duckdb
v2.0.0


## Add AK-2025 data to DB

In [6]:
tables = [
    ("NIBRS_incident.csv", "incident"),
    ("agencies.csv", "agencies"),
    ("NIBRS_OFFENSE.csv", "offense"),
    ("NIBRS_OFFENDER.csv", "offender"),
    ("NIBRS_VICTIM.csv", "victim"),
    ("NIBRS_VICTIM_OFFENSE.csv", "victim_offense"),
    ("NIBRS_VICTIM_OFFENDER_REL.csv", "victim_offender_rel")
]

def load_state_year(state, year):
    zip_path = data_dir / "nibrs" / str(year) / f"{state}-{year}.zip"
    extracted_path = data_dir / "extracted" / f"{state}-{year}"
    with zipfile.ZipFile(zip_path) as zipped:
        for csv_name, table_name in tables:
            csv_path = zipped.extract(csv_name, extracted_path)
            con.sql(f"""
                CREATE TABLE IF NOT EXISTS {table_name} AS
                SELECT *,
                '{state}' AS source_state,
                {year} AS source_year
                FROM read_csv('{csv_path}')
                LIMIT 0
            """)
            con.sql(f"""
                DELETE FROM {table_name}
                WHERE source_state = '{state}' AND source_year = {year}
            """)
            con.sql(f"""
                INSERT INTO {table_name} BY NAME
                SELECT *,
                '{state}' AS source_state,
                {year} AS source_year
                FROM read_csv('{csv_path}')
            """)
            db_rows = con.sql(f"""
                SELECT count(*) 
                FROM {table_name} 
                WHERE source_state = '{state}' 
                    AND source_year = {year}
            """).fetchall()[0][0]
            csv_rows = con.sql(f"SELECT count(*) FROM read_csv('{csv_path}')").fetchall()[0][0]
            assert db_rows == csv_rows, f"{table_name}: table {db_rows}, CSV {csv_rows}"
            print(f"{table_name}: {db_rows} rows")

load_state_year("AK", 2025)

incident: 14717 rows
agencies: 32 rows
offense: 17136 rows
offender: 16042 rows
victim: 16878 rows
victim_offense: 18672 rows
victim_offender_rel: 10022 rows


## Duplicate checks

In [3]:
dupe_checks = [
    ("incident", "incident_id"),
    ("agencies", "agency_id"),
    ("offense", "offense_id"),
    ("offender", "offender_id"),
    ("victim", "victim_id"),
    ("victim_offense", "victim_id, offense_id"),
    ("victim_offender_rel", "victim_id, offender_id"),
]

for table_name, key in dupe_checks:
    dupes = con.sql(f"""
        SELECT {key}, count(*)
        FROM {table_name}
        GROUP BY {key}
        HAVING count(*) > 1
    """).fetchall()
    assert dupes == [], f"{table_name}: duplicate {key}: {dupes[:5]}"
    print(f"{table_name}: no duplicate {key}")

incident: no duplicate incident_id
agencies: no duplicate agency_id
offense: no duplicate offense_id
offender: no duplicate offender_id
victim: no duplicate victim_id
victim_offense: no duplicate victim_id, offense_id
victim_offender_rel: no duplicate victim_id, offender_id


## Add lookup tables to db

In [4]:
state, year = "AK", 2025

zip_path = data_dir / "nibrs" / str(year) / f"{state}-{year}.zip"
extracted_path = data_dir / "extracted" / f"{state}-{year}"

lookup_tables = [
    ("NIBRS_OFFENSE_TYPE.csv", "offense_type"),
    ("NIBRS_RELATIONSHIP.csv", "relationship"),
    ("NIBRS_VICTIM_TYPE.csv", "victim_type"),
    ("NIBRS_AGE.csv", "age")
]
with zipfile.ZipFile(zip_path) as zipped:
    for csv_name, table_name in lookup_tables:
        csv_path = zipped.extract(csv_name, extracted_path)
        con.sql(f"""
            CREATE OR REPLACE TABLE {table_name} AS
            SELECT *
            FROM read_csv('{csv_path}')
        """)
        db_rows = con.sql(f"SELECT count(*) FROM {table_name}").fetchall()[0][0]
        csv_rows = con.sql(f"SELECT count(*) FROM read_csv('{csv_path}')").fetchall()[0][0]
        assert db_rows == csv_rows, f"{table_name}: table {db_rows}, CSV {csv_rows}"
        print(f"{table_name}: {db_rows} rows")

offense_type: 86 rows
relationship: 30 rows
victim_type: 9 rows
age: 104 rows


## Join checks

In [5]:
join_checks = [
    # (child table, column, parent table, column)
    ("offender", "incident_id", "incident", "incident_id"),
    ("victim", "incident_id", "incident", "incident_id"),
    ("offense", "incident_id", "incident", "incident_id"),
    ("incident", "agency_id", "agencies", "agency_id"),
    ("victim_offense", "victim_id", "victim", "victim_id"),
    ("victim_offense", "offense_id", "offense", "offense_id"),
    ("victim_offender_rel", "victim_id", "victim", "victim_id"),
    ("victim_offender_rel", "offender_id", "offender", "offender_id"),
    ("offense", "offense_code", "offense_type", "offense_code"),
    ("victim", "victim_type_id", "victim_type", "victim_type_id"),
    ("victim", "age_id", "age", "age_id"),
    ("offender", "age_id", "age", "age_id"),
    ("victim_offender_rel", "relationship_id", "relationship", "relationship_id"),
]

for child, child_col, parent, parent_col in join_checks:
    orphans = con.sql(f"""
        SELECT COUNT(*)
        FROM {child} c
        LEFT JOIN {parent} p ON c.{child_col} = p.{parent_col}
        WHERE p.{parent_col} IS NULL
    """).fetchall()[0][0]
    assert orphans == 0, f"{child}.{child_col}: {orphans} rows not found in {parent}"
    print(f"{child}.{child_col} -> {parent}: ok")

offender.incident_id -> incident: ok
victim.incident_id -> incident: ok
offense.incident_id -> incident: ok
incident.agency_id -> agencies: ok
victim_offense.victim_id -> victim: ok
victim_offense.offense_id -> offense: ok
victim_offender_rel.victim_id -> victim: ok
victim_offender_rel.offender_id -> offender: ok
offense.offense_code -> offense_type: ok
victim.victim_type_id -> victim_type: ok
victim.age_id -> age: ok
offender.age_id -> age: ok
victim_offender_rel.relationship_id -> relationship: ok
